In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import  GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score , classification_report


from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [2]:
START_DATE = "2015-01-01"

raw_df = yf.download("BTC-USD", start=START_DATE, auto_adjust=True, progress=False)

if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

raw_df = raw_df.dropna()
raw_df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,314.248993,320.434998,314.002991,320.434998,8036550
2015-01-02,315.032013,315.838989,313.565002,314.079010,7860650
2015-01-03,281.082001,315.149994,281.082001,314.846008,33054400
2015-01-04,264.195007,287.230011,257.612000,281.145996,55629100
2015-01-05,274.473999,278.341003,265.084015,265.084015,43962800


In [3]:
df = raw_df.copy()
#new
df["open_close"] = df["Open"] - df["Close"]
df["low_high"] = df["Low"] - df["High"]

df["pct_change"] = df["Close"].pct_change()
df["volume_change"] = df["Volume"].pct_change()



In [4]:
df = df.dropna()
df.head(3)

Price,Close,High,Low,Open,Volume,open_close,low_high,pct_change,volume_change
Date,,,,,,,,,
2015-01-02,315.032013,315.838989,313.565002,314.079010,7860650,-0.953003,-2.273987,0.002492,-0.021888
2015-01-03,281.082001,315.149994,281.082001,314.846008,33054400,33.764008,-34.067993,-0.107767,3.205047
2015-01-04,264.195007,287.230011,257.612000,281.145996,55629100,16.950989,-29.618011,-0.060079,0.682956


In [5]:
df["target_next_close"] = df["Close"].shift(-1)
df = df.dropna()

FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume", "open_close", "low_high",
                 "pct_change", "volume_change"]
TARGET_COL = "target_next_close"

X = df[FEATURE_COLS]
y = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)


In [6]:
X.shape, y.shape

((4272, 9), (4272,))

In [7]:
from  sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,shuffle=False,random_state=42)


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(objective="binary:logistic",
                          eval_metric = "logloss",
                          tree_method="hist",
                          random_state=RANDOM_STATE,
                          device='cuda'))
])

param_grid = {
    "xgb__n_estimators": [50,100,200,400],
    "xgb__max_depth": [3, 5, 7, 10],
    "xgb__learning_rate": [0.001, 0.01, 0.05, 0.1, 1],
    "xgb__subsample": [0.5 , 0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    verbose=10,
)

grid_search.fit(X_train, y_train)

print("بهترین params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Fitting 5 folds for each of 240 candidates, totalling 1200 fits
[CV 1/5; 1/240] START xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [09:04:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV 1/5; 1/240] END xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5;, score=0.529 total time=   0.7s
[CV 2/5; 1/240] START xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5
[CV 2/5; 1/240] END xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5;, score=0.529 total time=   0.2s
[CV 3/5; 1/240] START xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5
[CV 3/5; 1/240] END xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5;, score=0.529 total time=   0.1s
[CV 4/5; 1/240] START xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5
[CV 4/5; 1/240] END xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5;, score=0.531 total time=   0.1s
[CV 5/5; 1/240] START xgb__learning_rate=0.001, xgb__max_depth=3, xgb__n_estimators=50, xgb__subsample=0.5
[CV 5/5; 1/240] END 

In [9]:
-grid_search.best_score_

np.float64(-0.5296566834633724)

In [10]:
grid_search.best_estimator_

Pipeline(steps=[('scaler', StandardScaler()),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device='cuda',
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.001, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=3,
                               max_leaves=None, min_child_weight=None,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=50,
                               n_jobs=None, num_parallel_tree=None, ...))])

In [11]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

f1 = f1_score(y_test, y_pred)
preci = precision_score(y_test, y_pred, zero_division=0)

print(f"f1:  {f1}")
print(f"precision_score: {preci}")


f1:  0.6477093206951027
precision_score: 0.47897196261682246


In [12]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       223
           1       0.48      1.00      0.65       205

    accuracy                           0.48       428
   macro avg       0.24      0.50      0.32       428
weighted avg       0.23      0.48      0.31       428



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
